##### Inporting Lybraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime 

##### Importing Dataset

In [2]:
# Importing data and dropping null values

df = pd.read_excel('Online%20Retail.xlsx')
df = df.dropna(subset=['CustomerID', 'Description'])
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


#### Product Performance Matrix

##### Calculating Product Matrics

In [37]:
# creating new column called 'Total_revenue'

df['Total_revenue'] = df['Quantity'] * df['UnitPrice']

In [ ]:
product_performance = df.groupby(['StockCode', 'Description']).agg({
    'Total_revenue': 'sum',
    'Quantity': 'sum',
    'InvoiceNo': 'nunique', # number of transactions
    'InvoiceDate': ['min', 'max'] # first and last sales
}).reset_index()

# Flatten column names

product_performance.columns = ['StockCode', 'Description', 'Total_revenue', 'Total_Quantity', 'Num_Transactions', 'First_Sale_Date', 'Last_Sale_Date']
product_performance.head()

,StockCode,Description,Total_revenue,Total_Quantity,Num_Transactions,First_Sale_Date,Last_Sale_Date
0,10002,INFLATABLE POLITICAL GLOBE,699.55,823,49,2010-12-01 08:45:00,2011-04-18 12:56:00
1,10080,GROOVY CACTUS INFLATABLE,114.41,291,21,2011-02-27 13:47:00,2011-11-21 17:04:00
2,10120,DOGGY RUBBER,40.53,193,29,2010-12-03 11:19:00,2011-12-04 13:15:00
3,10125,MINI FUNKY DESIGN TAPES,930.30,1226,61,2010-12-01 12:23:00,2011-12-09 10:13:00
4,10133,COLOURING PENCILS BROWN TUBE,1143.61,2384,122,2010-12-01 12:15:00,2011-09-07 10:07:00


##### Analyze sales velocity (units sold per time period)

In [ ]:
# changing date format to 'YYYY.MM.DD'

product_performance['First_Sale_Date'] = pd.to_datetime(product_performance['First_Sale_Date'].dt.strftime('%Y.%m.%d'))
product_performance['Last_Sale_Date'] = pd.to_datetime(product_performance['Last_Sale_Date'].dt.strftime('%Y.%m.%d'))

# creating column called 'Days_activity' from the substraction of 'Last_Sale_Date' and 'First_Sale_Date'

product_performance['Days_activity'] = (pd.to_datetime(product_performance['Last_Sale_Date']) - pd.to_datetime(product_performance['First_Sale_Date'])).dt.days + 1
product_performance['Days_activity'] = product_performance['Days_activity'].astype(int)

# calculating sales velocity (Unit per day)

product_performance['Sales_velocity'] = (product_performance['Total_Quantity'] / product_performance['Days_activity']).round(2)
product_performance.head()

,StockCode,Description,Total_revenue,Total_Quantity,Num_Transactions,First_Sale_Date,Last_Sale_Date,Days_activity,Sales_velocity
0,10002,INFLATABLE POLITICAL GLOBE,699.55,823,49,2010-12-01,2011-04-18,139,5.92
1,10080,GROOVY CACTUS INFLATABLE,114.41,291,21,2011-02-27,2011-11-21,268,1.09
2,10120,DOGGY RUBBER,40.53,193,29,2010-12-03,2011-12-04,367,0.53
3,10125,MINI FUNKY DESIGN TAPES,930.30,1226,61,2010-12-01,2011-12-09,374,3.28
4,10133,COLOURING PENCILS BROWN TUBE,1143.61,2384,122,2010-12-01,2011-09-07,281,8.48


##### Top/Bottom Product Analysis

In [59]:
# Top 10 products by Total Revenue ($)

top_10_revenue = product_performance.nlargest(10, 'Total_revenue')[['StockCode',	'Description',	'Total_revenue']]

print(" |Top 10 Products by Total Revenue: |")
top_10_revenue.reset_index()

 |Top 10 Products by Total Revenue: |


,index,StockCode,Description,Total_revenue
0,2529,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60
1,1245,22423,REGENCY CAKESTAND 3 TIER,142592.95
2,3576,85123A,WHITE HANGING HEART T-LIGHT HOLDER,100448.15
3,3569,85099B,JUMBO BAG RED RETROSPOT,85220.78
4,2027,23166,MEDIUM CERAMIC TOP STORAGE JAR,81416.73
5,3896,POST,POSTAGE,77803.96
6,2607,47566,PARTY BUNTING,68844.33
7,2810,84879,ASSORTED COLOUR BIRD ORNAMENT,56580.34
8,3894,M,Manual,53779.93
9,1933,23084,RABBIT NIGHT LIGHT,51346.20


In [60]:
# Bottom 10 products by Total Revenue

bottom_10_revenue = product_performance[product_performance['Total_revenue'] > 0].nsmallest(10, 'Total_revenue')[['StockCode',	'Description',	'Total_revenue']]

print(" Bottom 10 Products by Total Revenue:")
bottom_10_revenue.reset_index()

 Bottom 10 Products by Total Revenue:


,index,StockCode,Description,Total_revenue
0,3895,PADS,PADS TO MATCH ALL CUSHIONS,0.003
1,2711,84227,HEN HOUSE W CHICK IN NEST,0.420
2,2261,23366,SET 12 COLOURING PENCILS DOILEY,0.650
3,398,21268,VINTAGE BLUE TINSEL REEL,0.840
4,2944,90084,PINK CRYSTAL GUITAR PHONE CHARM,0.850
5,2958,90104,PURPLE FRANGIPANI HAIRCLIP,0.850
6,3315,84201C,HAPPY BIRTHDAY CARD TEDDY/CAKE,0.950
7,3317,84206B,CAT WITH SUNGLASSES BLANK CARD,0.950
8,2840,84990,60 GOLD AND SILVER FAIRY CAKE CASES,1.100
9,2267,23370,SET 36 COLOURING PENCILS DOILEY,1.250


In [64]:
product_performance['Movment_category'] = pd.qcut(product_performance['Sales_velocity'], 3, labels=['slow_moving', 'medium_moving', 'fast_moving'])
product_performance.head()

,StockCode,Description,Total_revenue,Total_Quantity,Num_Transactions,First_Sale_Date,Last_Sale_Date,Days_activity,Sales_velocity,Movment_category
0,10002,INFLATABLE POLITICAL GLOBE,699.55,823,49,2010-12-01,2011-04-18,139,5.92,fast_moving
1,10080,GROOVY CACTUS INFLATABLE,114.41,291,21,2011-02-27,2011-11-21,268,1.09,medium_moving
2,10120,DOGGY RUBBER,40.53,193,29,2010-12-03,2011-12-04,367,0.53,slow_moving
3,10125,MINI FUNKY DESIGN TAPES,930.30,1226,61,2010-12-01,2011-12-09,374,3.28,medium_moving
4,10133,COLOURING PENCILS BROWN TUBE,1143.61,2384,122,2010-12-01,2011-09-07,281,8.48,fast_moving


##### Inventory Analysis:

##### Inventory Turnover Analysis

In [ ]:
# Extarcting months from 'InvoiceDate' for turnover analysis

df['Year_month'] = df['InvoiceDate'].dt.to_period('M')

# Monthly sales by product category

monthly_sales = df.groupby(['Year_month', 'Description'])\
    .agg({
        'Quantity': 'sum',
        'Total_revenue': 'sum'
    }).reset_index()

monthly_sales

,Year_month,Description,Quantity,Total_revenue
0,2010-12,4 PURPLE FLOCK DINNER CANDLES,14,35.70
1,2010-12,OVAL WALL MIRROR DIAMANTE,9,89.55
2,2010-12,SET 2 TEA TOWELS I LOVE LONDON,257,758.15
3,2010-12,10 COLOUR SPACEBOY PEN,547,464.95
4,2010-12,12 COLOURED PARTY BALLOONS,48,31.20
...,...,...,...,...
30647,2011-12,ZINC T-LIGHT HOLDER STAR LARGE,85,81.88
30648,2011-12,ZINC T-LIGHT HOLDER STARS SMALL,120,100.40
30649,2011-12,ZINC WILLIE WINKIE CANDLE STICK,135,117.09
30650,2011-12,ZINC WIRE KITCHEN ORGANISER,16,63.20


In [ ]:
# Monthly sales analysis for turnover

monthly_turnover = monthly_sales.groupby('Description')\
    .agg({
        'Quantity': ['mean', 'std']
    }).reset_index()
monthly_turnover.columns = ['Description', 'AvgMonthlySales', 'SalesStdDev']
monthly_turnover.head()

,Description,AvgMonthlySales,SalesStdDev
0,4 PURPLE FLOCK DINNER CANDLES,11.666667,16.977704
1,50'S CHRISTMAS GIFT BAG LARGE,377.000000,394.449617
2,DOLLY GIRL BEAKER,399.666667,359.477213
3,I LOVE LONDON MINI BACKPACK,90.000000,63.566238
4,I LOVE LONDON MINI RUCKSACK,1.000000,NaN


In [86]:
 # Coefficient of variation

monthly_turnover['Coeff_of_Variation'] = (monthly_turnover['SalesStdDev'] / monthly_turnover['AvgMonthlySales']).round(2)
monthly_turnover.head()

,Description,AvgMonthlySales,SalesStdDev,Coeff_of_Variation
0,4 PURPLE FLOCK DINNER CANDLES,11.666667,16.977704,1.46
1,50'S CHRISTMAS GIFT BAG LARGE,377.000000,394.449617,1.05
2,DOLLY GIRL BEAKER,399.666667,359.477213,0.90
3,I LOVE LONDON MINI BACKPACK,90.000000,63.566238,0.71
4,I LOVE LONDON MINI RUCKSACK,1.000000,NaN,NaN
